In [1]:
import pandas as pd
import math
from collections import Counter

# Charger les données
data = pd.read_csv("../data/diabetes.csv")
# On définit la colonne target (ce qu'on veut prédire)
# ici c'est Outcome (0 ou 1)
target_col = data.columns[-1]
#Les features = toutes les colonnes sauf la dernière
features = list(data.columns[:-1])

# Fonction sigmoid
#La sigmoid est nécessaire pour transformer la sortie continue du modèle en probabilité
def sigmoid(x):
    return 1 / (1 + math.exp(-x))


# Initialisation prédictions
#je crée une nouvelle colonne appelée y_hat et je mets 0 partout au début”
data["y_hat"] = 0.0


# Calcul gradient et hessian
# cette fonction sert à mesurer les erreurs du modèle
def compute_gradients(data):
    # liste pour stocker les erreurs (gradient)
    gradients = []

    # liste pour stocker la "force" de l'erreur (hessian)
    hessians = []

    # on parcourt toutes les lignes du dataset
    for i in range(len(data)):
        # vraie valeur (0 ou 1)
        y = data.iloc[i][target_col]
        # prédiction actuelle du modèle (log-odds)
        y_hat = data.iloc[i]["y_hat"]
        # on transforme y_hat en probabilité avec sigmoid
        p = sigmoid(y_hat)

        # GRADIENT
        # erreur simple : prédiction - vrai
        # si positif → on a trop prédit
        # si négatif → on a sous-estimé
        g = p - y
        # HESSIAN
        # mesure la "sensibilité" de l'erreur
        # plus p est proche de 0.5 → plus c’est incertain
        h = p * (1 - p)

        # on stocke les valeurs
        gradients.append(g)
        hessians.append(h)

    # on ajoute dans le dataset deuxx colonnes g et h 
    # chaque ligne a maintenant son erreur
    data["g"] = gradients
    data["h"] = hessians

# Split des données
def split_data(data, feature, split_value):
    #Ici on prend toutes les lignes où la valeur de la colonne (feature)
    # est inférieure ou égale au split value
    # ça forme le groupe gauche (left)
    left = data[data[feature] <= split_value]
    # Ici on prend toutes les lignes où la valeur de la colonne
    # est strictement supérieure au split value
    # ça forme le groupe droite (right)
    right = data[data[feature] > split_value]
    # On retourne les deux groupes
    return left, right


# cette fonction permet de mesurer si un split (division)
# elle sert à décider si une séparation des données améliore ou non le modèle
# est intéressant ou pas pour construire l'arbre
def calc_gain(G_left, H_left, G_right, H_right, lam=1):
    # fonction interne qui calcule le score d'un groupe
    def score(G, H):
        # G = somme des gradients (erreurs)
        # H = somme des hessians (importance des erreurs)

        # formule utilisée dans XGBoost
        # plus le résultat est grand → meilleur groupe
        return (G**2) / (H + lam)

    # somme des gradients des deux côtés
    G = G_left + G_right
    # somme des hessians des deux côtés
    H = H_left + H_right
    
    # calcul du gain
    # score après séparation (gauche + droite)
    left_right_score = score(G_left, H_left) + score(G_right, H_right)

    # score si on ne coupe pas (tout ensemble)
    parent_score = score(G, H)
    # gain = amélioration apportée par la division
    return left_right_score - parent_score

# Trouver meilleur split
# cette fonction cherche la meilleure colonne + valeur
# pour séparer les données en 2 groupes (gauche / droite)
def best_split(data):
    # on commence avec un très mauvais score
    # (comme ça n’importe quel split sera meilleur au début)
    best_gain = -1
    # on stocke la meilleure colonne (feature)
    best_feature = None
    # on stocke la meilleure valeur de séparation
    best_value = None
    # on teste chaque colonne (feature)
    for feature in features:

        # on récupère toutes les valeurs possibles de cette colonne
        values = sorted(data[feature].unique())

        # on teste chaque valeur comme seuil
        for v in values:
            # on divise le dataset en 2 groupes
            left, right = split_data(data, feature, v)

            # si un groupe est vide alors on ignore
            # car ce n’est pas un vrai split utile
            if len(left) == 0 or len(right) == 0:
                continue
                
            # on calcule les erreurs (g et h)
            # somme des gradients côté gauche
            G_left = left["g"].sum()
            # somme des hessians côté gauche
            H_left = left["h"].sum()
            # somme des gradients côté droite
            G_right = right["g"].sum()
            # somme des hessians côté droite
            H_right = right["h"].sum()

            # on calcule le gain de ce split
            gain = calc_gain(G_left, H_left, G_right, H_right)
            # on garde le meilleur split
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_value = v

    # on retourne le meilleur choix trouvé
    return best_feature, best_value

# Calcul valeur feuille
def leaf_value(data, lam=1):
    # on calcule la somme des gradients (erreurs)
    G = data["g"].sum()

    # on calcule la somme des hessians (importance des erreurs)
    H = data["h"].sum()

    # formule de XGBoost pour une feuille
    # on calcule la correction à appliquer (valeur que l’arbre va ajouter à la prédiction)
    return - G / (H + lam)
    

# CONSTRUCTION DE L'ARBRE
# cette fonction construit un arbre de décision récursivement
def build_tree(data, depth=0, max_depth=2):

    # on arrête si :
    # on a atteint la profondeur max
    # ou s’il reste trop peu de données
    if depth == max_depth or len(data) < 5:

        # on retourne une feuille (valeur finale)
        return leaf_value(data)
        
    # CHOISIR LA MEILLEURE COUPE
    feature, split_value = best_split(data)
    # si aucun bon split trouvé
    if feature is None:
        # on retourne une feuille directement
        return leaf_value(data)
        
    # DIVISER LES DONNÉES
    left, right = split_data(data, feature, split_value)

    # CONSTRUIRE L'ARBRE RÉCURSIVEMENT
    return {
        "feature": feature,          # colonne utilisée pour couper
        "split_value": split_value,  # valeur de séparation
        # sous-arbre gauche (on continue à diviser)
        "left": build_tree(left, depth+1, max_depth),
        # sous-arbre droit (on continue à diviser)
        "right": build_tree(right, depth+1, max_depth)
    }


# Prédiction avec un arbre de décision
def predict_tree(tree, row):
    # si on arrive à une feuille (valeur finale)
    # ça veut dire que l’arbre a terminé sa décision
    #cette condition vérifie si on est arrivé à une feuille de l’arbre,qui contient une valeur finale de prédiction
    if isinstance(tree, float):
        return tree
    # on récupère la colonne utilisée pour la décision
    feature = tree["feature"]
    # on récupère la valeur de séparation
    split_value = tree["split_value"]
    # on regarde la valeur de la ligne pour cette colonne
    # si elle est plus petite ou égale au seuil
    if row[feature] <= split_value:
        # on va vers le sous-arbre gauche
        return predict_tree(tree["left"], row)
    else:
        # sinon on va vers le sous-arbre droit
        return predict_tree(tree["right"], row)


# Entraînement du modèle XGBoost
def train_xgboost(data, n_estimators=5, learning_rate=0.1):
    # liste qui va contenir tous les arbres
    trees = []
    # on va construire plusieurs arbres un par un
    for i in range(n_estimators):
        # on calcule les erreurs du modèle actuel
        # (gradient + hessian)
        compute_gradients(data)
        # on construit un arbre pour corriger ces erreurs
        tree = build_tree(data)
        # on ajoute l'arbre dans la forêt
        trees.append(tree)
        
        # mise à jour des prédictions
        # pour chaque ligne du dataset
        for j in range(len(data)):
            # on récupère la ligne
            row = data.iloc[j]
            # on fait une prédiction avec le nouvel arbre
            update = predict_tree(tree, row)
            # on ajoute cette correction à la prédiction actuelle
            data.at[j, "y_hat"] += learning_rate * update

    # on retourne tous les arbres construits
    return trees


# Prédiction finale avec XGBoost
def predict_xgboost(trees, row, learning_rate=0.1):
    # on commence avec une prédiction vide
    y_hat = 0
    # on parcourt tous les arbres du modèle
    for tree in trees:
        # chaque arbre donne une petite correction
        # on ajoute cette correction à la prédiction globale
        y_hat += learning_rate * predict_tree(tree, row)
    # on transforme le score en probabilité
    # sigmoid ramène la valeur entre 0 et 1
    p = sigmoid(y_hat)
    # décision finale
    # si probabilité > 0.5 → classe 1
    # sinon → classe 0
    return 1 if p > 0.5 else 0


# TEST DU MODÈLE
# on entraîne le modèle XGBoost (on construit plusieurs arbres)
trees = train_xgboost(data.copy(), n_estimators=5)
# message pour dire que le modèle est prêt
print("Model trained!")
# compteur pour les bonnes prédictions
correct = 0
# on parcourt tout le dataset pour tester le modèle
for i in range(len(data)):
    # on prend une ligne du dataset (un patient)
    row = data.iloc[i]
    # on fait une prédiction avec le modèle
    pred = predict_xgboost(trees, row)
    # si la prédiction est correcte
    if pred == row[target_col]:
        correct += 1

# calcul de la précision
accuracy = correct / len(data)
# affichage du résultat
print("Accuracy:", accuracy)

Model trained!
Accuracy: 0.78125
